# Notebook 9b: AlphaEarth-Based Fine-Resolution Regression Downscaling

Third super-resolution method, structurally different from notebook 9's
guided filter and residual injection: instead of using a stale MODIS VCF
guide raster from a different year, train a regression model directly on
(fine AlphaEarth embedding features, fine real MODIS VCF value) pairs from
the guide tiles, then apply it to any year's AlphaEarth data -- no guide
raster needed at inference time at all, which removes the "guide is from a
different year" problem entirely.

Reuses the exact same region-stratified guide/test tile split as notebook 9
(`GUIDE_TILES`/`TEST_TILES` pasted below verbatim from that notebook's run),
so results are comparable across all three methods on identical tiles.

**GCS access code (Cells 4-6) is copied near-verbatim from
`3za-metrics_alphaearth_gcs.ipynb`**, which is already validated (ran
successfully across all 69 production tiles with a passing internal
consistency sanity check). The only real adaptation is `fetch_alphaearth_fine_gcs()`
in Cell 6: 3za warps AlphaEarth onto an intermediate fine grid
(`TILE_SIZE=600 * SUBSAMPLE_FACTOR=8 = 4800`) before block-reducing it down
to min/mean/median/max per 2km cell. That intermediate grid is *already*
exactly 4800x4800 -- the same size as `TILE_SIZE_MODIS` here -- so this
notebook just keeps that grid directly instead of reducing it further.

**Memory warning:** that fine grid is 4800x4800x64 bands x 4 bytes (float32)
= ~5.9GB per tile, held transiently in memory during each tile's extraction
(`build_training_rows()` masks down to valid pixels immediately afterward,
so it doesn't accumulate across tiles -- but the single-tile peak is real).
Test on 2-3 tiles before committing to a full 46-guide-tile run.


In [ ]:
# ## Cell 1: CONFIGURATION
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

from pathlib import Path

INFERENCE_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/inference")
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/superres")
PRODUCT_OUTPUT_DIR = OUTPUT_DIR / "products"
TRAINING_DIR = OUTPUT_DIR / "alphaearth_regression_training"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
PRODUCT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODIS_REFERENCE_YEAR = 2020
EMBEDDING_YEAR = 2020  # matches notebook 3za's fixed proxy year

TILE_SIZE_MODIS = 4800  # fine (250m) grid -- also 3za's intermediate warp grid size
NO_DATA_OUT = 255       # matches write_vcf_geotiff's uint8 convention elsewhere in the pipeline

# AlphaEarth GCS access constants -- identical to 3za-metrics_alphaearth_gcs.ipynb
AEF_BUCKET = "alphaearth_foundations"
AEF_PREFIX_BASE = "satellite_embedding/v1/annual"
AEF_NODATA_RAW = -128
N_EMBEDDING_BANDS = 64
EMBEDDING_BAND_NAMES = [f"A{i:02d}" for i in range(N_EMBEDDING_BANDS)]
HEADER_CHECK_WORKERS = 24

# MODIS sinusoidal projection parameters (matches the rest of the pipeline)
MODIS_SPHERE_RADIUS = 6371007.181
MODIS_TILE_SIZE_M = 1111950.5196666666
MODIS_UPPER_LEFT_X = -20015109.354
MODIS_UPPER_LEFT_Y = 10007554.677
SINU_PROJ4 = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"

STANDARD_TEST_TILES = ["h09v05", "h12v04", "h12v09", "h20v06", "h31v11"]

# Per-tile random subsample cap for the GUIDE-tile training accumulation --
# without this, concatenating every valid fine pixel from all 46 guide
# tiles could reach ~500M+ rows (~135GB as float32), which does not fit
# in memory. Test-tile evaluation (Cell 9) does NOT use this cap -- it
# scores every valid pixel per tile for an honest per-tile RMSE.
MAX_PIXELS_PER_TILE = 200_000

# Same region-stratified guide/test split as notebook 9 (pasted verbatim from
# its Cell 4 output) -- reused so all three super-resolution methods are
# compared on identical tiles.
GUIDE_TILES = ['h08v04', 'h08v05', 'h09v04', 'h09v05', 'h10v05', 'h10v06', 'h11v04', 'h11v09', 'h11v10', 'h12v01', 'h12v02', 'h12v03', 'h12v04', 'h12v09', 'h12v12', 'h13v02', 'h13v10', 'h13v11', 'h16v01', 'h17v05', 'h18v03', 'h18v04', 'h19v04', 'h19v07', 'h19v10', 'h19v12', 'h20v03', 'h20v06', 'h20v08', 'h20v09', 'h20v10', 'h20v11', 'h21v01', 'h21v02', 'h21v06', 'h21v10', 'h22v04', 'h23v03', 'h24v02', 'h24v04', 'h26v06', 'h27v04', 'h28v11', 'h29v11', 'h29v12', 'h30v12']

TEST_TILES = ['h10v04', 'h11v02', 'h11v03', 'h11v05', 'h11v08', 'h12v05', 'h12v10', 'h13v01', 'h13v12', 'h18v07', 'h19v08', 'h19v09', 'h19v11', 'h20v02', 'h20v04', 'h21v04', 'h21v05', 'h22v03', 'h23v02', 'h24v03', 'h27v06', 'h27v07', 'h31v11']

print("Configuration loaded")
print(f"  Guide tiles: {len(GUIDE_TILES)}, Test tiles: {len(TEST_TILES)}")


In [ ]:
# ## Cell 2: Imports

import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import requests
import rasterio
import rasterio.transform
import rasterio.crs
from osgeo import gdal, osr
import xgboost as xgb
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

print("Imports complete")


In [ ]:
# ## Cell 3: Projection + GCS + UTM Helpers
# (copied near-verbatim from 3za-metrics_alphaearth_gcs.ipynb Cells 3-4)

def get_tile_bounds_sinusoidal(tile: str):
    """Tile bounds in sinusoidal coordinates (min_x, min_y, max_x, max_y)."""
    h = int(tile[1:3])
    v = int(tile[4:6])
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_x = min_x + MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    min_y = max_y - MODIS_TILE_SIZE_M
    return (min_x, min_y, max_x, max_y)


def sinusoidal_to_latlon(x, y):
    lat = np.degrees(y / MODIS_SPHERE_RADIUS)
    lon = np.degrees(x / (MODIS_SPHERE_RADIUS * np.cos(np.radians(lat))))
    return lat, lon


def get_tile_latlon_bounds(tile: str):
    """Approximate lat/lon bounding box for a tile, with a small buffer."""
    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)
    corners_x = [min_x, max_x, min_x, max_x]
    corners_y = [min_y, min_y, max_y, max_y]
    lats, lons = [], []
    for x, y in zip(corners_x, corners_y):
        lat, lon = sinusoidal_to_latlon(x, y)
        lats.append(lat)
        lons.append(lon)
    buffer = 1.0
    return min(lats) - buffer, max(lats) + buffer, min(lons) - buffer, max(lons) + buffer


def get_sinusoidal_srs():
    srs = osr.SpatialReference()
    srs.ImportFromProj4(SINU_PROJ4)
    return srs


SINU_WKT = get_sinusoidal_srs().ExportToWkt()


def gcs_https_url(gs_path: str) -> str:
    """gs://bucket/key -> https://storage.googleapis.com/bucket/key (public,
    provider-pays -- no auth needed for this bucket)."""
    assert gs_path.startswith("gs://"), gs_path
    return "https://storage.googleapis.com/" + gs_path[len("gs://"):]


def latlon_to_utm_zone(lat: float, lon: float) -> str:
    """Standard UTM zone (e.g. '10N'). Ignores Norway/Svalbard special cases,
    which don't affect the tiles this pipeline processes."""
    lon = ((lon + 180) % 360) - 180
    zone_number = int((lon + 180) / 6) + 1
    zone_number = max(1, min(60, zone_number))
    hemisphere = "N" if lat >= 0 else "S"
    return f"{zone_number}{hemisphere}"


def utm_zones_for_tile(tile: str):
    """UTM zones potentially covering this tile -- sampled across a grid of
    points over the tile's lat/lon bbox, since a ~1112km MODIS tile can span
    more than one 6-degree-wide UTM zone."""
    lat_min, lat_max, lon_min, lon_max = get_tile_latlon_bounds(tile)
    lats = np.linspace(lat_min, lat_max, 5)
    lons = np.linspace(lon_min, lon_max, 5)
    zones = set()
    for lat in lats:
        for lon in lons:
            zones.add(latlon_to_utm_zone(lat, lon))
    return zones


def list_gcs_objects(bucket: str, prefix: str):
    """List object names under a prefix in a public GCS bucket via the
    anonymous JSON API (no credentials needed), handling pagination."""
    objects = []
    page_token = None
    while True:
        params = {"prefix": prefix, "maxResults": 1000}
        if page_token:
            params["pageToken"] = page_token
        resp = requests.get(
            f"https://storage.googleapis.com/storage/v1/b/{bucket}/o",
            params=params, timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()
        objects.extend(item["name"] for item in data.get("items", []))
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    return objects

print("Projection + GCS + UTM helper functions defined")


In [ ]:
# ## Cell 4: Find Covering COGs
# (copied verbatim from 3za-metrics_alphaearth_gcs.ipynb Cell 5)

def _cog_overlap_url(obj_name: str, min_x: float, min_y: float, max_x: float, max_y: float):
    """Open one candidate COG's header and return its URL if it overlaps the
    target sinusoidal bounds, else None. Builds its own SRS/transform objects
    rather than sharing them across threads (GDAL/OSR objects aren't safe to
    share that way)."""
    url = gcs_https_url(f"gs://{AEF_BUCKET}/{obj_name}")
    ds = gdal.Open(f"/vsicurl/{url}")
    if ds is None:
        return None

    gt = ds.GetGeoTransform()
    src_srs_wkt = ds.GetProjection()
    w, h = ds.RasterXSize, ds.RasterYSize
    ds = None

    src_min_x = gt[0]
    src_max_x = gt[0] + w * gt[1]
    src_max_y = gt[3]
    src_min_y = gt[3] + h * gt[5]

    src_srs = osr.SpatialReference()
    src_srs.ImportFromWkt(src_srs_wkt)
    tile_srs = get_sinusoidal_srs()
    transform = osr.CoordinateTransformation(src_srs, tile_srs)

    corners = [
        (src_min_x, src_min_y), (src_max_x, src_min_y),
        (src_max_x, src_max_y), (src_min_x, src_max_y),
    ]
    xs, ys = [], []
    for x, y in corners:
        px, py, _ = transform.TransformPoint(x, y)
        xs.append(px)
        ys.append(py)

    file_min_x, file_max_x = min(xs), max(xs)
    file_min_y, file_max_y = min(ys), max(ys)

    overlaps = not (
        file_max_x < min_x or file_min_x > max_x or
        file_max_y < min_y or file_min_y > max_y
    )
    return url if overlaps else None


def find_covering_cogs(tile: str, embedding_year: int):
    """List candidate COGs in each UTM zone/year folder the tile might touch,
    then open each candidate's own header concurrently, keeping only the
    ones whose real bounds (reprojected into our sinusoidal CRS) actually
    overlap the tile."""
    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)
    zones = utm_zones_for_tile(tile)
    logger.info(f"  UTM zones to search: {sorted(zones)}")

    covering = []
    for zone in zones:
        prefix = f"{AEF_PREFIX_BASE}/{embedding_year}/{zone}/"
        try:
            objects = list_gcs_objects(AEF_BUCKET, prefix)
        except Exception as e:
            logger.warning(f"    {zone}: listing failed ({e})")
            continue

        tiff_objects = [o for o in objects if o.endswith(".tiff")]
        logger.info(f"    {zone}: {len(tiff_objects)} candidate file(s)")

        with ThreadPoolExecutor(max_workers=HEADER_CHECK_WORKERS) as pool:
            futures = [
                pool.submit(_cog_overlap_url, obj_name, min_x, min_y, max_x, max_y)
                for obj_name in tiff_objects
            ]
            for future in as_completed(futures):
                url = future.result()
                if url is not None:
                    covering.append(url)

    logger.info(f"  Found {len(covering)} covering COG(s) for {tile}")
    return covering

print("Covering-COG search function defined (concurrent header checks)")


In [ ]:
# ## Cell 5: Fetch Fine-Resolution AlphaEarth Embeddings
# Adapted from 3za's fetch_alphaearth_tile_gcs() -- identical warp + de-quantize
# steps, but returns the full (64, 4800, 4800) fine grid directly instead of
# block-reducing it down to min/mean/median/max per 2km cell.

def fetch_alphaearth_fine_gcs(tile: str, embedding_year: int = EMBEDDING_YEAR):
    """
    Warp all covering COGs (nearest-neighbor, so raw quantized values are
    preserved exactly) onto the tile's full 4800x4800 (250m) sinusoidal
    grid, then de-quantize (exact formula from the AlphaEarth GCS docs).

    Returns (64, 4800, 4800) float32 in native embedding units
    (approximately [-1, 1]), NaN for NoData, or None if no covering COGs
    were found.
    """
    cog_urls = find_covering_cogs(tile, embedding_year)
    if not cog_urls:
        logger.warning(f"  No AlphaEarth COG coverage for {tile} in {embedding_year}")
        return None

    min_x, min_y, max_x, max_y = get_tile_bounds_sinusoidal(tile)

    vsicurl_paths = [f"/vsicurl/{u}" for u in cog_urls]
    warp_opts = gdal.WarpOptions(
        format="MEM",
        outputBounds=(min_x, min_y, max_x, max_y),
        width=TILE_SIZE_MODIS,
        height=TILE_SIZE_MODIS,
        dstSRS=SINU_WKT,
        srcNodata=AEF_NODATA_RAW,
        dstNodata=AEF_NODATA_RAW,
        resampleAlg="near",
        multithread=True,
    )
    ds = gdal.Warp("", vsicurl_paths, options=warp_opts)
    if ds is None:
        logger.warning(f"  Warp failed for {tile}")
        return None

    raw = ds.ReadAsArray().astype(np.float32)  # (64, 4800, 4800)
    ds = None

    valid = raw != AEF_NODATA_RAW
    dequant = np.where(valid, ((raw / 127.5) ** 2) * np.sign(raw), np.nan)
    del raw, valid
    return dequant

print("Fine-resolution AlphaEarth fetch function defined")


In [ ]:
# ## Cell 6: Build Training Table from Guide Tiles

def load_fine_reference(tile: str) -> np.ndarray:
    """MODIS_VCF_C6_{year}_{tile}_250m.tif -> (4800, 4800) float32, NaN for NoData."""
    path = INFERENCE_DIR / f"MODIS_VCF_C6_{MODIS_REFERENCE_YEAR}_{tile}_250m.tif"
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
    data[data == NO_DATA_OUT] = np.nan
    return data


def build_training_rows(tile: str, max_pixels: int = None, seed: int = 42):
    """Returns (X, y): X is (n_rows, 64) AlphaEarth embedding features,
    y is (n_rows,) real MODIS VCF percent tree cover, at valid fine pixels.
    Masks down to valid pixels immediately so the ~5.9GB full-tile embedding
    array doesn't persist in memory.

    If max_pixels is given and there are more valid pixels than that, a
    random (seeded) subsample of max_pixels is kept -- use this for training
    accumulation across many tiles (see MAX_PIXELS_PER_TILE), but leave it
    None for evaluation, so per-tile RMSE reflects every valid pixel."""
    embeddings = fetch_alphaearth_fine_gcs(tile)
    if embeddings is None:
        return None
    target = load_fine_reference(tile)

    valid = np.isfinite(target) & np.all(np.isfinite(embeddings), axis=0)
    n_valid = int(np.sum(valid))
    if n_valid == 0:
        del embeddings
        return None

    X = embeddings[:, valid].T
    y = target[valid]
    del embeddings

    if max_pixels is not None and len(y) > max_pixels:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(y), size=max_pixels, replace=False)
        X, y = X[idx], y[idx]

    return X, y

print("Training-table builder defined")


In [ ]:
# ## Cell 7: Extract Training Data from Guide Tiles (checkpointed -- resumable)

for tile in GUIDE_TILES:
    out_path = TRAINING_DIR / f"{tile}_alphaearth_train.npz"
    if out_path.exists():
        logger.info(f"  {tile}: already extracted, skipping ({out_path.name})")
        continue

    result = build_training_rows(tile, max_pixels=MAX_PIXELS_PER_TILE)
    if result is None:
        logger.warning(f"  {tile}: no usable training data, skipped")
        continue
    X, y = result
    np.savez_compressed(out_path, X=X.astype(np.float32), y=y.astype(np.float32))
    logger.info(f"  {tile}: {len(y):,} training pixels -> saved {out_path.name}")

print("Extraction complete (or resumed) for all available guide tiles")


In [ ]:
# ## Cell 7b: Load All Saved Per-Tile Training Data

all_X, all_y = [], []
for tile in GUIDE_TILES:
    path = TRAINING_DIR / f"{tile}_alphaearth_train.npz"
    if not path.exists():
        continue
    data = np.load(path)
    all_X.append(data["X"])
    all_y.append(data["y"])

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
print(f"Loaded {len(y_all):,} total training pixels from {len(all_X)} tiles")


In [ ]:
# ## Cell 8: Train XGBoost Regressor

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.15, random_state=42)

model = xgb.XGBRegressor(
    tree_method="hist", n_estimators=500, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print("Model trained")


In [ ]:
# ## Cell 9: Final Evaluation on Test Tiles (checkpointed -- resumable)

import json  # defensive re-import in case Cell 2 wasn't re-run this kernel session
import os

print("=" * 70)
print("ALPHAEARTH REGRESSION -- FINAL EVALUATION ON TEST TILES")
print("=" * 70)

EVAL_DIR = OUTPUT_DIR / "alphaearth_regression_eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

alphaearth_test_results = []
for tile in TEST_TILES:
    metrics_path = EVAL_DIR / f"{tile}_alphaearth_eval.json"
    if metrics_path.exists():
        try:
            with open(metrics_path) as f:
                m = json.load(f)
            alphaearth_test_results.append(m)
            print(f"  {tile:<8} (cached) RMSE={m['rmse']:6.2f}%  MAE={m['mae']:6.2f}%  "
                  f"bias={m['bias']:+6.2f}%  correlation_r2={m['correlation_r2']:.4f}  n={m['n_valid']:,}")
            continue
        except (json.JSONDecodeError, KeyError):
            logger.warning(f"  {tile}: cached file is corrupt/incomplete ({metrics_path.name}), re-extracting")
            # fall through and re-extract below

    result = build_training_rows(tile)
    if result is None:
        logger.warning(f"  {tile}: no usable data, skipped")
        continue
    X_tile, y_tile = result
    pred = model.predict(X_tile)
    residual = pred - y_tile
    r_pearson, _ = pearsonr(y_tile, pred)
    m = {
        "tile": tile,
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "mae": float(np.mean(np.abs(residual))),
        "bias": float(np.mean(residual)),
        "correlation_r2": float(r_pearson ** 2),
        "n_valid": int(len(y_tile)),
    }
    # Write atomically: build the file under a temp name, then rename it into
    # place. A crash mid-write leaves the temp file corrupt, never the real
    # checkpoint path -- so a later run can never mistake a partial write for
    # a completed one (which is exactly what happened here before this fix).
    tmp_path = metrics_path.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(m, f)
    os.replace(tmp_path, metrics_path)
    print(f"  {tile:<8} RMSE={m['rmse']:6.2f}%  MAE={m['mae']:6.2f}%  bias={m['bias']:+6.2f}%  "
          f"correlation_r2={m['correlation_r2']:.4f}  n={m['n_valid']:,}")
    alphaearth_test_results.append(m)

overall_rmse_ae = float(np.mean([m["rmse"] for m in alphaearth_test_results]))
print(f"\nMean RMSE: {overall_rmse_ae:.2f}%")
print("(Compare against notebook 9's baseline/guided-filter/residual-injection RMSE on the same TEST_TILES)")


In [ ]:
# ## Cell 10: Product GeoTIFFs -- AlphaEarth Regression (5 standard test tiles)
# Writes into the SAME products/ directory notebook 9's Cell 15 uses, so all
# three methods' output tifs for these tiles sit side by side.

def get_fine_transform(tile: str):
    h = int(tile[1:3])
    v = int(tile[4:6])
    min_x = MODIS_UPPER_LEFT_X + h * MODIS_TILE_SIZE_M
    max_y = MODIS_UPPER_LEFT_Y - v * MODIS_TILE_SIZE_M
    pixel_size = MODIS_TILE_SIZE_M / TILE_SIZE_MODIS
    return rasterio.transform.from_origin(min_x, max_y, pixel_size, pixel_size)


def write_fine_geotiff(data: np.ndarray, output_path: Path, description: str):
    out_data = np.where(np.isfinite(data) & (data >= 0) & (data <= 100),
                         np.round(data).astype(np.uint8), NO_DATA_OUT)
    with rasterio.open(
        output_path, "w", driver="GTiff",
        height=data.shape[0], width=data.shape[1], count=1,
        dtype=np.uint8, crs=get_sinusoidal_srs().ExportToWkt(),
        transform=get_fine_transform(tile), nodata=NO_DATA_OUT,
        compress="lzw", tiled=True,
    ) as dst:
        dst.write(out_data, 1)
        dst.set_band_description(1, description)


for tile in STANDARD_TEST_TILES:
    embeddings = fetch_alphaearth_fine_gcs(tile)
    if embeddings is None:
        logger.warning(f"{tile}: no AlphaEarth coverage, skipping product generation")
        continue

    valid_shape = embeddings.shape[1:]
    flat = embeddings.reshape(N_EMBEDDING_BANDS, -1).T
    finite_rows = np.all(np.isfinite(flat), axis=1)
    pred_flat = np.full(flat.shape[0], np.nan, dtype=np.float32)
    if finite_rows.any():
        pred_flat[finite_rows] = model.predict(flat[finite_rows])
    pred = pred_flat.reshape(valid_shape)
    del embeddings, flat, pred_flat

    write_fine_geotiff(pred, PRODUCT_OUTPUT_DIR / f"SuperRes_alphaearth_{tile}_250m.tif",
                        f"AlphaEarth regression {tile}")
    print(f"{tile}: wrote AlphaEarth-regression product")

print(f"\nProducts saved to: {PRODUCT_OUTPUT_DIR}")


In [ ]:
# ## Cell 11: Visualize AlphaEarth-Regression Prediction (same area as notebook 9a's example)
# Requires `model` to already exist (Cell 8 must have completed). Fetches a
# fresh full-tile AlphaEarth embedding for the tile (not cached from training,
# since test tiles are never accumulated into X_all/y_all) -- this can take
# 10-25+ minutes depending on how many UTM zones the tile spans.

import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter

PACE_YEAR = 2025
TILE_SIZE_PACE = 600
AGGREGATION_FACTOR = TILE_SIZE_MODIS // TILE_SIZE_PACE


def load_coarse_prediction(tile: str) -> np.ndarray:
    path = INFERENCE_DIR / f"PACE_VCF_{PACE_YEAR}_{tile}_2km.tif"
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
    data[data == NO_DATA_OUT] = np.nan
    return data


def upsample_nearest(coarse: np.ndarray, factor: int) -> np.ndarray:
    return np.repeat(np.repeat(coarse, factor, axis=0), factor, axis=1)


def find_high_contrast_crop(fine_reference: np.ndarray, crop_coarse_size: int = 8,
                             aggregation_factor: int = AGGREGATION_FACTOR):
    """Identical logic to notebook 9a's find_high_contrast_crop -- locates the
    highest-local-variance window in the real MODIS reference, so for the
    same tile this reproduces the same crop location as 9a's comparison plot."""
    window = crop_coarse_size * aggregation_factor

    valid = np.isfinite(fine_reference)
    if not valid.any():
        return None

    filled = np.where(valid, fine_reference, 0.0).astype(np.float32)
    valid_f = valid.astype(np.float32)

    sum_x = uniform_filter(filled, size=window, mode="constant") * window ** 2
    sum_x2 = uniform_filter(filled ** 2, size=window, mode="constant") * window ** 2
    count = uniform_filter(valid_f, size=window, mode="constant") * window ** 2

    with np.errstate(invalid="ignore", divide="ignore"):
        mean = sum_x / count
        local_var = sum_x2 / count - mean ** 2

    local_var = np.where(count >= window ** 2 * 0.9, local_var, -1.0)
    best_row_fine, best_col_fine = np.unravel_index(np.argmax(local_var), local_var.shape)
    crop_row = (best_row_fine - window // 2) // aggregation_factor
    crop_col = (best_col_fine - window // 2) // aggregation_factor
    return int(crop_row), int(crop_col)


def plot_alphaearth_example(tile: str, crop_coarse_size: int = 8, crop_row: int = None, crop_col: int = None):
    coarse_pred = load_coarse_prediction(tile)
    fine_reference = load_fine_reference(tile)
    baseline = upsample_nearest(coarse_pred, AGGREGATION_FACTOR)

    if crop_row is None or crop_col is None:
        found = find_high_contrast_crop(fine_reference, crop_coarse_size)
        if found is not None:
            crop_row, crop_col = found
        else:
            crop_row = (TILE_SIZE_PACE - crop_coarse_size) // 2
            crop_col = (TILE_SIZE_PACE - crop_coarse_size) // 2
    crop_row = max(0, min(crop_row, TILE_SIZE_PACE - crop_coarse_size))
    crop_col = max(0, min(crop_col, TILE_SIZE_PACE - crop_coarse_size))

    fine_row0 = crop_row * AGGREGATION_FACTOR
    fine_row1 = (crop_row + crop_coarse_size) * AGGREGATION_FACTOR
    fine_col0 = crop_col * AGGREGATION_FACTOR
    fine_col1 = (crop_col + crop_coarse_size) * AGGREGATION_FACTOR

    # Fetch the full-tile embedding once, then predict only on the cropped
    # region -- avoids running the model over the full 4800x4800 grid.
    embeddings = fetch_alphaearth_fine_gcs(tile)
    if embeddings is None:
        raise RuntimeError(f"No AlphaEarth coverage for {tile}")
    emb_crop = embeddings[:, fine_row0:fine_row1, fine_col0:fine_col1]
    del embeddings

    flat = emb_crop.reshape(N_EMBEDDING_BANDS, -1).T
    finite_rows = np.all(np.isfinite(flat), axis=1)
    pred_flat = np.full(flat.shape[0], np.nan, dtype=np.float32)
    if finite_rows.any():
        pred_flat[finite_rows] = model.predict(flat[finite_rows])
    ae_crop = np.clip(pred_flat.reshape(fine_row1 - fine_row0, fine_col1 - fine_col0), 0, 100)

    coarse_crop = coarse_pred[crop_row:crop_row + crop_coarse_size, crop_col:crop_col + crop_coarse_size]
    baseline_crop = baseline[fine_row0:fine_row1, fine_col0:fine_col1]
    ref_crop = fine_reference[fine_row0:fine_row1, fine_col0:fine_col1]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    im0 = axes[0].imshow(coarse_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[0].set_title(f"{tile}: Coarse prediction (2km)\n{crop_coarse_size}x{crop_coarse_size} PACE pixels")
    im1 = axes[1].imshow(baseline_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[1].set_title("Baseline (nearest upsample)")
    im2 = axes[2].imshow(ae_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[2].set_title("AlphaEarth regression")
    im3 = axes[3].imshow(ref_crop, vmin=0, vmax=100, cmap="YlGn", interpolation="nearest")
    axes[3].set_title(f"Real MODIS C6 {MODIS_REFERENCE_YEAR} (250m)")
    for ax, im in zip(axes, [im0, im1, im2, im3]):
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="% Tree Cover")
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f"alphaearth_example_{tile}_crop_r{crop_row}_c{crop_col}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    logger.info(f"Saved: {fig_path}")
    plt.show()


plot_alphaearth_example(TEST_TILES[0])
